### data collection and processing
To aquire the necessary song lyric data, [Genius' API](https://docs.genius.com) is used, because it could provide access to all lyrics from the initial selection. The process of obtaining lyrics is fairly easy, because there is only a client access token needed, which can be aquired with a free user account. The access token is locally stored in a [file](./key.json). To send queries to the API, the [lyricsgenius](https://pypi.org/project/lyricsgenius/) module is used, it returns the lyrics for a provided artist and songtitle as a string, which is then processed and stored together with other information for each song in a separate [file](./songlist_with_lyrics.json).

In [4]:
import re
import json
import string
import lyricsgenius
from retry import retry

In [ ]:
# open the json-file containing all songs categorized by year and keyed with artist and title
with open('test_songs.json', 'r') as file:
    data = json.load(file)

In [6]:
# write an empty json file for all songs to be analized
with open('songlist_with_lyrics.json', 'w') as f:
    json.dump({'songs':[]}, f, indent=4)

In [7]:
with open('key.json', 'r') as file:
    genius_api = lyricsgenius.Genius(json.load(file)['apikey'])

In [8]:
# API call, queried by artist and title of each song
@retry(delay=5)
def fetch_from_api(artist, title):
    try:
        lyrics = genius_api.search_song(artist, title).lyrics
    except:
        print(f'ERROR: Timed out while fetching {title} by {artist} - trying it again...')
        raise
    return lyrics

In [9]:
# strip linebreaks '\n', punctuation and wrong words (like part names of songs) from song text provided by the API
def process_lyrics(lyrics):
    processed_lyrics = lyrics.lower()
    processed_lyrics = re.sub('\\[(.*?)\\]', '', processed_lyrics)

    translation_table_punct = str.maketrans('', '', string.punctuation)
    translation_table_linebreaks = str.maketrans('\n', ' ')
    processed_lyrics = processed_lyrics.translate(translation_table_punct).translate(translation_table_linebreaks)

    processed_lyrics = processed_lyrics.lstrip()
    processed_lyrics = processed_lyrics.replace('   ', ' ')
    processed_lyrics = processed_lyrics.replace('  ', ' ')
    return processed_lyrics

In [10]:
# function to append the json-string containing the lyrics to the already existing entries
def write_to_file(per_song_data):
    with open('songlist_with_lyrics.json', 'r') as f:
        file_data = json.load(f)
        file_data['songs'].append(per_song_data)

    with open('songlist_with_lyrics.json', 'w') as f:
        json.dump(file_data, f, indent=4)

In [11]:
# looping through the songs per year
for year in data:
    for song in range(len(data[year])):
        artist = data[year][song]['artist']
        title = data[year][song]['title']

        # append year to song's existing data
        year_dict = {'year': f"{year}"}
        data_per_song = data[year][song]
        data_per_song.update(year_dict)

        lyrics = fetch_from_api(artist, title)    
        
        # clean up the data from the API and update the content of the json-string
        lyrics = process_lyrics(lyrics)

        # lyrics = lyrics.encode('unicode_escape').decode('unicode_escape')

        # append the lyrics to the song's existing data
        lyrics_dict = {'lyrics': f"{lyrics}"}
        data_per_song.update(lyrics_dict)

        # write to 'songlist_with_lyrics.json'
        write_to_file(data_per_song)

ERROR: Timed out while fetching Billie Jean by Michael Jackson - trying it again...
ERROR: Timed out while fetching Billie Jean by Michael Jackson - trying it again...
ERROR: Timed out while fetching Billie Jean by Michael Jackson - trying it again...
ERROR: Timed out while fetching Billie Jean by Michael Jackson - trying it again...


### lyrics analysis and categorization
During this process every song is analysed